# Notebook 1 · Flat Multi-Agent System

> **Series:** LangChain MAS Architectures · Travel Agency Toy Use Case

---

## What is a Flat Architecture?

In a **flat MAS** all agents operate at the *same authority level*.
There is no manager, no hierarchy, and no fixed execution order.
Agents share a common workspace (a "board") and each one can read and modify it freely.

```
┌──────────────────────────────────────────────────────┐
│                    SHARED BOARD                      │
│  (visible and writable by every peer at all times)   │
└──────┬───────────────┬────────────────┬──────────────┘
       │               │                │
  ┌────▼────┐     ┌────▼────┐     ┌─────▼────┐
  │  Peer A │◄───►│  Peer B │◄───►│  Peer C  │
  │Dest/Time│     │Booking  │     │Activities│
  └─────────┘     └─────────┘     └──────────┘
        peer-to-peer, no authority gradient
```

### Key Properties
| Property | Flat |
|---|---|
| Authority | Equal — all peers |
| Communication | Peer → shared board |
| Stop condition | Peers agree (democratic) |
| Failure mode | Can loop or contradict without a tiebreaker |

### When to Use It
A flat design suits situations where no single agent should own the decision,
where diversity of opinion matters, or where you want emergent consensus
rather than top-down instruction.

---

## What We Will Build

Three peer agents collaborate on the same travel request:

- **Destination peer** — focuses on *where* and *when*
- **Booking peer** — focuses on *flight* and *hotel*
- **Experience peer** — focuses on *activities* and overall trip balance

> Their specialties are **tendencies, not restrictions**.
> Any peer can challenge or refine any part of the plan — that is the
> defining feature of a flat topology.


## 1 · Setup: One Model, One Request, One Catalog

The same three blocks appear in every notebook of this series:

| Block | Purpose |
|---|---|
| `model` | One small, cheap LLM used by every agent |
| `USER_REQUEST` | The single traveler request we want to fulfill |
| `CATALOG` + helpers | The only data source agents are allowed to use |

Keeping these identical lets you compare orchestration patterns in isolation.
Nothing changes between notebooks except *how agents talk to each other*.

> **Prerequisite:** set `OPENAI_API_KEY` in your environment before running.


In [1]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

# ── One model powers every agent in the series ──────────────────────────────
# We use gpt-4.1-mini for speed and low cost during learning.
# Swap to any model that supports structured output (temperature=0 keeps
# outputs deterministic so you can rerun and compare results).
model = init_chat_model("openai:gpt-4.1-mini", temperature=0)

# ── The traveler request ─────────────────────────────────────────────────────
# All four notebooks solve this exact request.
# The only thing that changes is the orchestration architecture.
USER_REQUEST = """\
Plan a 4-day spring trip from Rome.
Requirements:
- mid-range budget
- easy flights
- central hotel
- mix of food and culture
- simple daily plan""".strip()

# ── Static catalog ────────────────────────────────────────────────────────────
# Agents must only use destinations, flights, hotels, and activities
# listed below. This constraint keeps the toy example clean and comparable.

DESTINATIONS = {
    "Lisbon":    {"best_period": "April–June", "style": "sunny, walkable, relaxed",
                  "notes": "great for food, viewpoints, and compact neighborhoods"},
    "Barcelona": {"best_period": "April–June", "style": "lively, artistic, seaside",
                  "notes": "strong mix of architecture, beach walks, and tapas"},
    "Prague":    {"best_period": "April–June", "style": "historic, compact, lower-cost",
                  "notes": "easy sightseeing with a classic old-town atmosphere"},
}

FLIGHTS = [
    {"destination": "Lisbon",    "code": "TP-833", "price": 180, "type": "direct",  "duration": "3h 05m"},
    {"destination": "Lisbon",    "code": "IB-310", "price": 150, "type": "1 stop",  "duration": "5h 10m"},
    {"destination": "Barcelona", "code": "VY-611", "price": 140, "type": "direct",  "duration": "1h 50m"},
    {"destination": "Barcelona", "code": "IB-220", "price": 125, "type": "1 stop",  "duration": "4h 00m"},
    {"destination": "Prague",    "code": "FR-721", "price": 110, "type": "direct",  "duration": "1h 55m"},
    {"destination": "Prague",    "code": "OS-410", "price": 145, "type": "1 stop",  "duration": "3h 45m"},
]

HOTELS = [
    {"destination": "Lisbon",    "name": "Baixa Stay",         "price_per_night": 145, "style": "central boutique hotel"},
    {"destination": "Lisbon",    "name": "River Rooms",         "price_per_night": 120, "style": "simple hotel near transit"},
    {"destination": "Barcelona", "name": "Born Hotel",          "price_per_night": 160, "style": "central design hotel"},
    {"destination": "Barcelona", "name": "Gracia Inn",          "price_per_night": 130, "style": "quiet hotel in a walkable district"},
    {"destination": "Prague",    "name": "Old Town House",      "price_per_night": 115, "style": "historic hotel near main sights"},
    {"destination": "Prague",    "name": "City Garden Hotel",   "price_per_night":  95, "style": "budget-friendly hotel with tram access"},
]

ACTIVITIES = [
    {"destination": "Lisbon",    "name": "Alfama food walk",                    "tag": "food",    "price": 35},
    {"destination": "Lisbon",    "name": "Belém and river tram day",            "tag": "culture", "price": 25},
    {"destination": "Barcelona", "name": "Gothic Quarter tapas evening",        "tag": "food",    "price": 40},
    {"destination": "Barcelona", "name": "Sagrada Família and modernism route", "tag": "culture", "price": 32},
    {"destination": "Prague",    "name": "Old Town walking tour",               "tag": "culture", "price": 18},
    {"destination": "Prague",    "name": "Czech dinner and jazz night",         "tag": "food",    "price": 30},
]

# ── Catalog helpers ───────────────────────────────────────────────────────────
# Simple filter functions so agents can look up options by destination.

def flights_for(destination: str) -> list[dict]:
    return [f for f in FLIGHTS if f["destination"] == destination]

def hotels_for(destination: str) -> list[dict]:
    return [h for h in HOTELS if h["destination"] == destination]

def activities_for(destination: str) -> list[dict]:
    return [a for a in ACTIVITIES if a["destination"] == destination]

def catalog_text() -> str:
    """Return the full catalog as a plain-text string suitable for a prompt."""
    lines = []
    for dest, info in DESTINATIONS.items():
        lines.append(f"Destination: {dest}")
        lines.append(f"  Best period : {info['best_period']}")
        lines.append(f"  Style       : {info['style']}")
        lines.append(f"  Notes       : {info['notes']}")
        lines.append("  Flights:")
        for f in flights_for(dest):
            lines.append(f"    - {f['code']} | {f['type']} | EUR {f['price']} | {f['duration']}")
        lines.append("  Hotels:")
        for h in hotels_for(dest):
            lines.append(f"    - {h['name']} | EUR {h['price_per_night']}/night | {h['style']}")
        lines.append("  Activities:")
        for a in activities_for(dest):
            lines.append(f"    - {a['name']} | {a['tag']} | EUR {a['price']}")
        lines.append("")
    return "\n".join(lines).strip()

CATALOG_TEXT = catalog_text()

print("USER_REQUEST:")
print(USER_REQUEST)
print("\nCatalog loaded — destinations:", list(DESTINATIONS.keys()))


USER_REQUEST:
Plan a 4-day spring trip from Rome.
Requirements:
- mid-range budget
- easy flights
- central hotel
- mix of food and culture
- simple daily plan

Catalog loaded — destinations: ['Lisbon', 'Barcelona', 'Prague']


## 2 · Define the Shared Schema and the Peer Agents

### Why use structured output here?

Each peer produces a `PeerTurn` object with exactly two fields:

- `update` — one plain-text contribution to the shared board
- `ready` — a boolean vote on whether the group can stop

Using `with_structured_output` forces every peer to return the same shape,
which makes the orchestration loop trivial to write and easy to follow.


In [2]:
# ── Schema ────────────────────────────────────────────────────────────────────
# PeerTurn is intentionally tiny.
# The `ready` field lets the loop terminate by peer consensus instead of
# relying on a fixed iteration count (which would be the flat architecture
# making an authoritative decision nobody voted on).

class PeerTurn(BaseModel):
    update: str  = Field(description="One concise contribution to the shared travel board.")
    ready:  bool = Field(description="True when this peer believes the plan is good enough to stop.")

# Bind the schema to the model so .invoke() returns a PeerTurn object directly.
peer_llm = model.with_structured_output(PeerTurn)

# ── Peer registry ─────────────────────────────────────────────────────────────
# A plain dict is all we need.
# Key   = peer name (used in board entries so readers can trace decisions)
# Value = the peer's specialty (a soft nudge, not a hard restriction)
PEERS = {
    "Destination peer": (
        "Focus on destination choice and best travel period. "
        "You may challenge or refine any other part of the plan."
    ),
    "Booking peer": (
        "Focus on flight and hotel fit for budget and convenience. "
        "You may challenge or refine any other part of the plan."
    ),
    "Experience peer": (
        "Focus on activity mix and overall trip coherence. "
        "You may challenge or refine any other part of the plan."
    ),
}

# ── Peer runner ───────────────────────────────────────────────────────────────
def run_peer(peer_name: str, specialty: str, shared_board: list[str]) -> PeerTurn:
    """
    Call one peer agent.

    The critical flat-architecture property is visible here:
    every peer receives the *complete* shared board as context.
    No agent has privileged access to a private channel.
    """
    board_text = "\n".join(shared_board)  # render board as plain text for the prompt

    messages = [
        SystemMessage(content=(
            "You are a peer agent in a flat travel-agency MAS.\n"
            "There is no manager and no fixed order — you are equal to every other peer.\n"
            "You can refine, disagree with, or replace any earlier idea on the board.\n"
            "Only use destinations, flights, hotels, and activities from the catalog.\n\n"
            f"Your specialty (a tendency, not a restriction): {specialty}"
        )),
        HumanMessage(content=(
            f"Traveler request:\n{USER_REQUEST}\n\n"
            f"Catalog:\n{CATALOG_TEXT}\n\n"
            f"Shared board so far:\n{board_text}\n\n"
            "Add one useful contribution. Vote ready=True only if the plan "
            "covers destination, flight, hotel, and activities well enough."
        )),
    ]

    return peer_llm.invoke(messages)

print("Peer agents defined:", list(PEERS.keys()))


Peer agents defined: ['Destination peer', 'Booking peer', 'Experience peer']


## 3 · Run the Flat Conversation Loop

This is where the flat architecture becomes concrete.

### Orchestration rules (flat)
1. All peers take one turn per round.
2. After each round, every peer's contribution is appended to the shared board.
3. The conversation stops when **all** peers vote `ready=True` — a unanimous
   democratic agreement, not a manager's decision.
4. A plain `for` loop is enough — no graph library needed.

> **Notice:** the orchestration code is about 15 lines.
> That simplicity is possible because the flat pattern needs no routing logic.


In [3]:
# ── Shared board ──────────────────────────────────────────────────────────────
# The board starts with a single seed instruction.
# Every peer will read this and add to it.
shared_board = ["Start: build a travel plan that satisfies the traveler request."]

MAX_ROUNDS = 3  # safety cap so the notebook never runs forever

for round_num in range(1, MAX_ROUNDS + 1):
    print(f"\n--- Round {round_num} ---")
    ready_votes = []

    for peer_name, specialty in PEERS.items():
        # ── Each peer reads the full board and adds one entry ──────────────
        turn = run_peer(peer_name, specialty, shared_board)

        # Append the peer's contribution so later peers (and later rounds)
        # can see it immediately — this is the live shared-board effect.
        entry = f"Round {round_num} | {peer_name}: {turn.update}"
        shared_board.append(entry)
        ready_votes.append(turn.ready)

        print(f"  [{peer_name}] ready={turn.ready}")
        print(f"    {turn.update[:120]}...")

    # ── Consensus check ───────────────────────────────────────────────────
    # In a flat system, stopping requires agreement from ALL peers.
    # No single agent can force the group to stop (or continue).
    if all(ready_votes):
        print("\nAll peers voted ready. Stopping early.")
        break

# ── Final formatting step ─────────────────────────────────────────────────────
# This call is NOT a manager role — it is a pure formatting step.
# We pass the full board and ask the model to render it as a clean itinerary.
# The decisions were already made collectively on the board.
final_itinerary = model.invoke([
    SystemMessage(content=(
        "Turn the shared travel board into one clean final itinerary. "
        "This is only a formatting step — do not change any decisions already made."
    )),
    HumanMessage(content=(
        f"Traveler request:\n{USER_REQUEST}\n\n"
        f"Shared board:\n{'\n'.join(shared_board)}"
    )),
]).content

print("\n" + "="*60)
print("SHARED BOARD (full history)")
print("="*60)
for item in shared_board:
    print(" •", item)

print("\n" + "="*60)
print("FINAL ITINERARY")
print("="*60)
print(final_itinerary)



--- Round 1 ---
  [Destination peer] ready=False
    Recommend Lisbon for the 4-day spring trip from Rome. Best travel period April–June fits perfectly. Choose direct flight...
  [Booking peer] ready=False
    Consider Barcelona as an alternative. It offers a lively, artistic vibe with direct flight VY-611 at EUR 140, just under...
  [Experience peer] ready=True
    Prague is a strong contender for a 4-day spring trip from Rome, especially for a mid-range budget. Direct flight FR-721 ...

--- Round 2 ---
  [Destination peer] ready=True
    All three destinations—Lisbon, Barcelona, and Prague—fit the traveler's spring timing and mid-range budget well, with di...
  [Booking peer] ready=True
    I suggest finalizing Prague as the destination for the 4-day spring trip from Rome. It offers the best combination of bu...
  [Experience peer] ready=True
    I agree with choosing Prague for its compactness and affordability, but to enhance the food and culture mix, let's sugge...

All peers vot

## 4 · Key Takeaways

| What you saw | Why it matters |
|---|---|
| Every peer reads the **full** board | No information is hidden — the flat architecture is transparent by design |
| Peers can contradict each other | There is no authority to prevent it; convergence emerges from the loop |
| The loop stops by **peer vote** | Consensus replaces top-down control |
| No graph library was needed | A plain `for` loop is sufficient for a flat, acyclic pattern |

### How this differs from the other notebooks
- **vs Hierarchical (nb 2):** no manager dictates the workflow; peers self-organize
- **vs Society (nb 3):** peers build one shared plan rather than voting on pre-built packages
- **vs Team (nb 4):** roles are *soft tendencies*, not strict ownership boundaries

### When the flat pattern struggles
- **Contradictions** accumulate if peers keep disagreeing without a tiebreaker
- **Quality degrades** if a minority of weak peers can block consensus
- **Latency grows** because every agent runs every round, even when it has nothing new to add
